# Single-example pipeline — Colab fallback (exported from the marimo notebook)

Linear Jupyter/Colab export of `single_example_pipeline.py`. Run cells **top to bottom**.
marimo reactivity is inactive here (plain kernel), so it runs once with the **default example**;
to change the example, edit the relevant cell and re-run the cells below it.

The first code cell installs `marimo`; **Cell 0** then installs `gen_gec_errant` + `errant` +
the spaCy model (on Colab it `pip install`s from GitHub and mounts Drive). CPU works end-to-end.

In [ ]:
# Colab bootstrap: install marimo so the `import marimo as mo` cell below can run.
# (Cell 0 installs gen_gec_errant + errant + the spaCy model.)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "marimo"])

In [ ]:
import marimo as mo

# Single-example pipeline trace — gen → GEC → ERRANT → compare

One learner sentence, every stage laid bare, ending in a three-source comparison
against the **authentic learner** (the yardstick).

> **Sources (CONCEPTS.md canonical):**
> **authentic learner** — the real human L2 continuation, the yardstick ·
> **matched control** — the native-pretrained model (`gpt2`) ·
> **artificial learner (AL)** — the same architecture *fine-tuned on learner text*
> (`ft-gpt2-small`). The matched pair `ft-gpt2-small : gpt2` differs *only* by fine-tuning.
>
> The quantity of interest is **distance to the authentic learner's error-tag
> distribution**, concentrated in acquisition categories — *not* raw error counts.

## Cell 0 — Environment + install (idempotent)

In [ ]:
# -- Cell 0: make a fresh runtime (local venv OR a clean Colab) work with no manual pip.
# Everything shells out via subprocess (no !pip / no magics — this is marimo, not Jupyter).
import importlib
import os
import subprocess
import sys
from pathlib import Path

def _colab() -> bool:
    try:
        import google.colab  # noqa: F401

        return True
    except Exception:
        return False

IN_COLAB = _colab()

def _pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

def _find_repo_root():
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "pyproject.toml").exists() and (cand / "src" / "gen_gec_errant").exists():
            return cand
    return None

_log = []

def _ensure(import_name, pip_args, label=None):
    label = label or import_name
    try:
        importlib.import_module(import_name)
        _log.append(f"✓ {label}: already importable")
        return
    except Exception:
        _log.append(f"… {label}: installing (pip install {' '.join(pip_args)})")
        _pip(*pip_args)
        importlib.invalidate_caches()
        _log.append(f"✓ {label}: installed")

# marimo itself (belt-and-suspenders for the exported .ipynb Colab path).
_ensure("marimo", ["marimo"])

# The pipeline package. Editable install from the repo locally; git install on a bare Colab.
_repo_root = _find_repo_root()
try:
    importlib.import_module("gen_gec_errant")
    _log.append("✓ gen_gec_errant: already importable")
except Exception:
    if _repo_root is not None:
        _log.append(f"… gen_gec_errant: pip install -e {_repo_root}")
        _pip("-e", str(_repo_root))
        # An editable install writes src/ into a .pth file that Python reads only at
        # interpreter *startup*; a pip install inside THIS running process therefore
        # won't be importable until we add src/ to sys.path ourselves (invalidate_caches
        # does not re-read .pth files). A fresh `python` picks it up automatically.
        _src = str(_repo_root / "src")
        if _src not in sys.path:
            sys.path.insert(0, _src)
        _log.append(f"✓ gen_gec_errant: added {_src} to sys.path (in-process editable import)")
    elif IN_COLAB:
        _url = "git+https://github.com/berstearns/gen-gec-errant-public.git"
        _log.append(f"… gen_gec_errant: pip install {_url}")
        _pip(_url)
    else:
        raise RuntimeError(
            "Could not find the repo root (pyproject.toml + src/gen_gec_errant). "
            "Run marimo from the repository root, or set IN_COLAB."
        )
    importlib.invalidate_caches()
    importlib.import_module("gen_gec_errant")
    _log.append("✓ gen_gec_errant: importable after install")

# ERRANT is a declared dependency (pulled by -e .) but double-ensure it and its spaCy model.
_ensure("errant", ["errant"])
try:
    import spacy

    spacy.load("en_core_web_sm")
    _log.append("✓ en_core_web_sm: loadable")
except Exception:
    _log.append("… en_core_web_sm: python -m spacy download en_core_web_sm")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    _log.append("✓ en_core_web_sm: downloaded")

# Mount Google Drive on Colab (guarded so local runs skip it entirely).
if IN_COLAB:
    try:
        from google.colab import drive

        drive.mount("/content/drive", force_remount=False)
        _log.append("✓ Google Drive mounted at /content/drive")
    except Exception as _e:  # noqa: F841
        _log.append(f"⚠ Drive mount skipped: {_e}")

install_log = "\n".join(_log)

In [ ]:
# Heavy + pipeline imports live in their own cell. Referencing `install_log` forces
# marimo to order this AFTER Cell 0's install cell (marimo orders by data dependency),
# so `import gen_gec_errant` below never runs before the pip install on a fresh env.
_ = install_log
import gc

import matplotlib

matplotlib.use("Agg")  # headless-safe; marimo renders the figure object regardless
import matplotlib.pyplot as plt
import torch

# --- Reuse the REAL pipeline code (do not reimplement load / perplexity / GEC / ERRANT) ---
from gen_gec_errant.annotation.config import AnnotationConfig
from gen_gec_errant.annotation.runner import (
    ERRANTAnnotator,
    classify_errors_by_region,
)
from gen_gec_errant.colab import resolve_model_path
from gen_gec_errant.data_loader.runner import make_prompts
from gen_gec_errant.gec.config import GECConfig
from gen_gec_errant.gec.runner import load_gec_corrector
from gen_gec_errant.generation.config import GenerationParams, ModelConfig
from gen_gec_errant.generation.runner import (
    compute_perplexity,
    get_device,
    load_model,
)
from gen_gec_errant.preprocessing.runner import split_into_sentences
from gen_gec_errant.registry import MODEL_REGISTRY, PathConfig, PIPELINE_DEFAULTS

DEVICE = get_device("auto")

In [ ]:
def _versions():
    import errant  # noqa
    import spacy
    import transformers

    return {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "spacy": spacy.__version__,
        "errant": getattr(__import__("errant"), "__version__", "?"),
    }

_v = _versions()
_rows = "\n".join(f"| {k} | `{val}` |" for k, val in _v.items())
mo.md(
    f"""
    **Environment resolved.**

    - `is_colab()` → **{IN_COLAB}**
    - resolved device → **`{DEVICE}`** *(CPU works end-to-end — slow, a few minutes/example;
      GPU used automatically when present)*

    | package | version |
    |---|---|
    {_rows}

    <details><summary>install log</summary>

    ```
    {install_log}
    ```
    </details>
    """
)

### Shared helpers (config tables, word diff, region colouring, acquisition view)
*Canonical labels + the acquisition categories from CONCEPTS.md §1.2 live here.*

In [ ]:
# Canonical display labels for the three sources (CONCEPTS.md §1.1).
SOURCE_LABELS = {
    "authentic_learner": "authentic learner",
    "matched_control": "matched control",
    "artificial_learner": "artificial learner (AL)",
}
SOURCE_ORDER = ["authentic_learner", "matched_control", "artificial_learner"]

# SLA-diagnostic "acquisition categories" (CONCEPTS.md §1.2). "*" = any ERRANT operation (M/R/U).
ACQUISITION_CATS = [
    ("*:DET", "determiner"),
    ("*:VERB:FORM", "verb morphology"),
    ("R:VERB:TENSE", "tense"),
    ("R:VERB:SVA", "subject–verb agreement"),
    ("*:PREP", "preposition"),
    ("R:NOUN:NUM", "noun number"),
]

def tag_in_cat(tag: str, cat: str) -> bool:
    """Does an ERRANT tag fall in an acquisition category pattern (with '*' wildcard op)?"""
    if cat.startswith("*:"):
        suffix = cat[2:]
        parts = tag.split(":", 1)
        return len(parts) == 2 and parts[1] == suffix
    return tag == cat

def acquisition_view(counts: dict) -> dict:
    """Map an error-tag count dict onto the six acquisition categories."""
    out = {}
    for cat, _name in ACQUISITION_CATS:
        out[cat] = sum(n for tag, n in counts.items() if tag_in_cat(tag, cat))
    return out

def esc(s) -> str:
    import html as _h

    return _h.escape(str(s))

def as_html(html_str: str):
    """Render a raw-HTML string in marimo, robust across versions."""
    try:
        return mo.Html(html_str)
    except Exception:
        return mo.md(html_str)

def kv_table(title: str, d: dict, defaults: dict | None = None):
    """Render a dataclass-style config as a fields->values table, flagging non-default values."""
    rows = []
    for k, v in d.items():
        flag = ""
        if defaults is not None and k in defaults and defaults[k] != v:
            flag = f" <span style='color:#b26a00'>← differs from default `{esc(defaults[k])}`</span>"
        rows.append(f"| `{esc(k)}` | `{esc(v)}`{flag} |")
    body = "\n".join(rows)
    return mo.md(f"**{title}**\n\n| field | value |\n|---|---|\n{body}")

def word_diff_html(original: str, corrected: str) -> str:
    """Word-level diff: deletions struck through (red), insertions green."""
    import difflib

    aw, bw = original.split(), corrected.split()
    sm = difflib.SequenceMatcher(a=aw, b=bw)
    parts = []
    for op, i1, i2, j1, j2 in sm.get_opcodes():
        if op == "equal":
            parts.append(esc(" ".join(aw[i1:i2])))
        elif op == "delete":
            parts.append(
                f"<span style='background:#ffd6d6;text-decoration:line-through'>{esc(' '.join(aw[i1:i2]))}</span>"
            )
        elif op == "insert":
            parts.append(f"<span style='background:#d6f5d6'>{esc(' '.join(bw[j1:j2]))}</span>")
        elif op == "replace":
            parts.append(
                f"<span style='background:#ffd6d6;text-decoration:line-through'>{esc(' '.join(aw[i1:i2]))}</span>"
                f" <span style='background:#d6f5d6'>{esc(' '.join(bw[j1:j2]))}</span>"
            )
    return "<div style='line-height:1.9'>" + " ".join(parts) + "</div>"

## Cell 1 — Pick the example

In [ ]:
# Hardcoded fixture (Stage 0): full authentic-learner sentences copied VERBATIM from
# data/celva-2-sample.csv (the `text` column). Each has >=1 visible learner error.
# No file/dataset dependency is needed for the default run.
EXAMPLES = [
    {
        "id": "celva-01",
        "text": "This magnet must be cooled by liquid nitrogen and that is the large cercle where we go through when we do an MRI.",
    },
    {
        "id": "celva-02",
        "text": "So with all of that we can make a matrix, with the hydrogen density and then convert it in a picture.",
    },
    {
        "id": "celva-03",
        "text": "The world have many riches at discover even too and its not necessary to have a Apple vision pro in addition.",
    },
    {
        "id": "celva-04",
        "text": "To do that we use electromagnetic flashs whitch are at the resonating frequency of hydrogen.",
    },
    {
        "id": "celva-05",
        "text": "Now with detectors we can calculate the among of time the spin needs to point to up again.",
    },
]

example_dropdown = mo.ui.dropdown(
    options={f"{e['id']}: {e['text'][:60]}…": i for i, e in enumerate(EXAMPLES)},
    value=f"{EXAMPLES[0]['id']}: {EXAMPLES[0]['text'][:60]}…",
    label="Example (authentic-learner sentence)",
)
custom_text_area = mo.ui.text_area(
    value="",
    label="…or paste your OWN sentence (overrides the dropdown when non-empty)",
    full_width=True,
)
mo.vstack([example_dropdown, custom_text_area])

In [ ]:
# Resolve the single working sentence. A pasted paragraph is reduced to its first
# sentence (mirrors the batch data_loader's split_sentences=True behaviour).
_custom = custom_text_area.value.strip()
if _custom:
    chosen_source = "custom (pasted)"
    _raw = _custom
else:
    _idx = example_dropdown.value if example_dropdown.value is not None else 0
    chosen_source = EXAMPLES[_idx]["id"]
    _raw = EXAMPLES[_idx]["text"]

_sents = split_into_sentences(_raw)
chosen_text = _sents[0] if _sents else _raw
_note = ""
if len(_sents) > 1:
    _note = f"\n\n> ℹ︎ input had {len(_sents)} sentences; using the **first** one (single-sentence pipeline)."

mo.md(
    f"""
    **Chosen source:** `{chosen_source}`

    > {chosen_text}

    *(words: {len(chosen_text.split())}){_note}*
    """
)

## Cell 2 — STAGE 1 · data_loader: sentence → prompt + reference

`make_prompts` splits the authentic sentence into a **prompt** (shared prefix the
models continue) and a **reference** continuation. The reference **IS the authentic
learner leg** — same sentence, real human continuation, error-profiled by the same
instrument. The char boundary `len(prompt)` is what later separates *prompt-region*
(shared, excluded) from *generation-region* (the source's own) errors.

In [ ]:
# Stage-1 config (DataLoaderConfig-style knobs). prompt_ratio=0.5, min_prompt_words=5.
_cfg = {
    "prompt_ratio": 0.5,
    "min_prompt_words": 5,
    "split_sentences": True,
}
s1 = make_prompts([chosen_text], prompt_ratio=0.5, min_prompt_words=5)[0]
prompt = s1["prompt"]
reference = s1["reference"]
boundary = len(prompt)

_cfg_table = kv_table("DataLoaderConfig (this stage)", _cfg, PIPELINE_DEFAULTS["data_loader"])
_split_table = mo.md(
    f"""
    | leg | text | words |
    |---|---|---|
    | **prompt** (shared, dim) | <span style='color:#888'>{prompt}</span> | {len(prompt.split())} |
    | **reference = authentic learner** | <span style='background:#fff3cd'>{reference}</span> | {len(reference.split())} |

    - **char boundary `len(prompt)` = {boundary}** — errors at char < {boundary} are *prompt-region* (excluded from every cross-source claim); char ≥ {boundary} are *generation-region* (counted).
    - The **reference continuation IS the authentic learner leg** — same sentence, real human continuation.
    """
)
mo.vstack([_cfg_table, _split_table])

## Cell 3 — Model selection (the triad) — trivially switchable

The demo pair is `ft-gpt2-small : gpt2` (AL : matched control) — identical architecture
+ size, differing **only** by fine-tuning on the learner corpus (EFCAMDAT). Swapping in
another registry pair is a one-click change below, not a code edit.

In [ ]:
# One-click model switching. Defaults = the locked demo pair ft-gpt2-small : gpt2.
al_reg_dropdown = mo.ui.dropdown(
    options=list(MODEL_REGISTRY.keys()),
    value="ft-gpt2-small",
    label="artificial learner (AL) — registry key",
)
control_hf_text = mo.ui.text(
    value="gpt2",
    label="matched control — HuggingFace id (loads everywhere)",
    full_width=True,
)
al_ckpt_text = mo.ui.text(value="", label="AL checkpoint dir override (blank = auto-resolve)", full_width=True)
mo.vstack([al_reg_dropdown, control_hf_text, al_ckpt_text])

In [ ]:
# -- authentic learner: no model; its continuation = the Stage-1 reference.
# -- matched control: a native HF id (gpt2), loads everywhere.
control_cfg = ModelConfig(name="matched_control", hf_model_id=control_hf_text.value.strip() or "gpt2")

# -- artificial learner: resolve the fine-tuned checkpoint dir via the real PathConfig
#    (Colab Drive vs local), i.e. models_root / gdrive_subpath / checkpoint_subdir.
_reg = MODEL_REGISTRY[al_reg_dropdown.value]
_paths = PathConfig.for_colab() if IN_COLAB else PathConfig.for_local()
_auto_ckpt = _paths.model_gdrive_path(_reg)  # None for native-only registry entries

_override = al_ckpt_text.value.strip()
al_ckpt_path = Path(_override) if _override else _auto_ckpt

al_available = False
al_load_kind = "none"
al_cfg = None
if al_ckpt_path is not None and al_ckpt_path.exists():
    if (al_ckpt_path / "config.json").exists():
        # HF-format checkpoint dir -> load by pointing hf_model_id at the directory.
        al_available = True
        al_load_kind = "hf_dir"
        al_cfg = ModelConfig(
            name="artificial_learner",
            hf_model_id=str(al_ckpt_path),
            model_family=_reg.model_family,
            is_learner_tuned=False,
        )
    else:
        _pt = [p for p in al_ckpt_path.glob("*.pt")] + [p for p in al_ckpt_path.glob("*.bin")]
        if _pt:
            al_available = True
            al_load_kind = "state_dict"
            al_cfg = ModelConfig(
                name="artificial_learner",
                hf_model_id="gpt2",
                model_family=_reg.model_family,
                is_learner_tuned=True,
                checkpoint_path=str(_pt[0]),
            )

if al_available:
    al_banner = mo.md(
        f"""
        ✅ **AL checkpoint found** — `{al_ckpt_path}` (load kind: `{al_load_kind}`).
        The AL leg will run. Registry entry: `{al_reg_dropdown.value}` — {_reg.description}.
        """
    )
else:
    al_banner = mo.callout(
        mo.md(
            f"""
            **AL checkpoint not found at** `{al_ckpt_path}` — **AL leg skipped**;
            matched control + authentic learner still run. Point the override box above at a
            real checkpoint dir to enable it. (Expected artefacts: `config.json` + a weights
            file for an HF dir, or a `*.pt` / `*.bin` state-dict.)
            """
        ),
        kind="warn",
    )

mo.vstack(
    [
        mo.md(
            f"""
            | source | role | model |
            |---|---|---|
            | **authentic learner** | yardstick | *(no model — continuation = Stage-1 reference: "{reference[:40]}…")* |
            | **matched control** | native-pretrained, same size | `{control_cfg.hf_model_id}` |
            | **artificial learner (AL)** | fine-tuned | `{al_reg_dropdown.value}` → `{al_ckpt_path}` |
            """
        ),
        al_banner,
    ]
)

## Cell 2b — Generation parameters (§5: 1 → 50 tokens, stop at end of sentence)

`min_new_tokens=1`, `max_new_tokens=50` (capped at 50), plus a **sentence-stop** so each
continuation is exactly one sentence. **Notebook-only** — the batch pipeline defaults are
untouched. Tweak and the generation re-runs reactively.

In [ ]:
_g = PIPELINE_DEFAULTS["generation"]
min_tokens_ui = mo.ui.number(start=1, stop=50, step=1, value=1, label="min_new_tokens")
max_tokens_ui = mo.ui.number(start=1, stop=50, step=1, value=50, label="max_new_tokens (≤50)")
temp_ui = mo.ui.number(start=0.1, stop=2.0, step=0.05, value=float(_g["temperature"]), label="temperature")
topk_ui = mo.ui.number(start=0, stop=200, step=1, value=int(_g["top_k"]), label="top_k")
topp_ui = mo.ui.number(start=0.1, stop=1.0, step=0.01, value=float(_g["top_p"]), label="top_p")
rep_pen_ui = mo.ui.number(
    start=1.0, stop=2.0, step=0.05, value=float(_g["repetition_penalty"]), label="repetition_penalty"
)
do_sample_ui = mo.ui.checkbox(value=bool(_g["do_sample"]), label="do_sample")
seed_ui = mo.ui.number(start=0, stop=999999, step=1, value=42, label="seed")
mo.vstack(
    [
        mo.hstack([min_tokens_ui, max_tokens_ui, temp_ui, seed_ui], justify="start"),
        mo.hstack([topk_ui, topp_ui, rep_pen_ui, do_sample_ui], justify="start"),
    ]
)

In [ ]:
# §5 sentence-stop. This faithfully MIRRORS gen_gec_errant.generation.runner.generate_continuations
# (same tokenization + same model.generate args) and ADDS a HuggingFace StoppingCriteria that
# halts once the newly-generated continuation contains a sentence terminator. Kept in the
# notebook (not in src/) per SPEC §5 so no batch-pipeline source is touched. Designed for a
# single prompt (batch_size=1), which is how the notebook calls it.
def generate_with_sentence_stop(model, tokenizer, prompt_text, gen_params, device, terminators=(".", "!", "?")):
    from transformers import StoppingCriteria, StoppingCriteriaList

    inputs = tokenizer(
        [prompt_text], return_tensors="pt", padding=True, truncation=True, max_length=512
    ).to(device)
    prompt_len = int(inputs["attention_mask"].sum().item())

    class _SentenceStop(StoppingCriteria):
        def __call__(self, input_ids, scores=None, **kwargs):
            done = []
            for row in range(input_ids.shape[0]):
                text = tokenizer.decode(input_ids[row, prompt_len:], skip_special_tokens=True)
                done.append(any(t in text for t in terminators))
            return torch.tensor(done, dtype=torch.bool, device=input_ids.device)

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=gen_params.max_new_tokens,
            min_new_tokens=gen_params.min_new_tokens,
            temperature=gen_params.temperature,
            top_k=gen_params.top_k,
            top_p=gen_params.top_p,
            do_sample=gen_params.do_sample,
            num_return_sequences=gen_params.num_return_sequences,
            repetition_penalty=gen_params.repetition_penalty,
            pad_token_id=tokenizer.pad_token_id,
            stopping_criteria=StoppingCriteriaList([_SentenceStop()]),
        )
    gen_ids = out[0, prompt_len:]
    raw = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return raw, int(gen_ids.shape[0])

## Cell 4 — STAGE 2 · generation (per model source)

For **matched control** and **AL**: `load_model` → generate (1→50 tokens, sentence-stop)
→ trim to the first sentence via `split_into_sentences(...)[0]` → perplexity on the full
text. The **authentic learner** needs no generation — its continuation is the reference.
Each model is freed (`del; gc; empty_cache`) before the next loads — mirroring the batch
runner's memory discipline. Both the **raw** (pre-trim) and **final one-sentence** outputs
are shown so the sentence-stop's effect is visible.

In [ ]:
gen_params_used = GenerationParams(
    max_new_tokens=int(max_tokens_ui.value),
    min_new_tokens=int(min_tokens_ui.value),
    temperature=float(temp_ui.value),
    top_k=int(topk_ui.value),
    top_p=float(topp_ui.value),
    do_sample=bool(do_sample_ui.value),
    repetition_penalty=float(rep_pen_ui.value),
)

def _run_generation():
    results = {}
    # authentic learner: no model, continuation = reference.
    results["authentic_learner"] = {
        "continuation_raw": reference,
        "continuation": reference,
        "full_text": f"{prompt} {reference}",
        "n_new_tokens": None,
        "perplexity": None,
        "model_class": "— (no model; authentic human continuation)",
        "param_count": None,
    }

    _to_run = [("matched_control", control_cfg)]
    if al_available and al_cfg is not None:
        _to_run.append(("artificial_learner", al_cfg))

    for _name, _cfg in _to_run:
        torch.manual_seed(int(seed_ui.value))
        model, tok = load_model(_cfg, DEVICE)
        _cls = type(model).__name__
        _params = sum(p.numel() for p in model.parameters())
        raw, n_new = generate_with_sentence_stop(model, tok, prompt, gen_params_used, DEVICE)
        _sents = split_into_sentences(raw)
        trimmed = _sents[0] if _sents else raw
        full_text = f"{prompt} {trimmed}"
        ppl = compute_perplexity(model, tok, [full_text], batch_size=1, device=DEVICE)[0]
        results[_name] = {
            "continuation_raw": raw,
            "continuation": trimmed,
            "full_text": full_text,
            "n_new_tokens": n_new,
            "perplexity": ppl,
            "model_class": _cls,
            "param_count": _params,
        }
        # Free before loading the next model (mirror the batch runner).
        del model, tok
        gc.collect()
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
    return results

sources = _run_generation()

def _render():
    blocks = [
        kv_table(
            "GenerationParams (this stage — notebook-only sentence-stop)",
            {
                "min_new_tokens": gen_params_used.min_new_tokens,
                "max_new_tokens": gen_params_used.max_new_tokens,
                "temperature": gen_params_used.temperature,
                "top_k": gen_params_used.top_k,
                "top_p": gen_params_used.top_p,
                "do_sample": gen_params_used.do_sample,
                "repetition_penalty": gen_params_used.repetition_penalty,
                "seed": int(seed_ui.value),
            },
            {**PIPELINE_DEFAULTS["generation"], "seed": 42},
        )
    ]
    for _name in ["authentic_learner", "matched_control", "artificial_learner"]:
        if _name not in sources:
            continue
        d = sources[_name]
        _ppl = f"{d['perplexity']:.2f}" if d["perplexity"] is not None else "— (n/a)"
        _pc = f"{d['param_count']:,}" if d["param_count"] is not None else "—"
        _nt = d["n_new_tokens"] if d["n_new_tokens"] is not None else "—"
        blocks.append(
            mo.md(
                f"""
                ### {_name}
                - **loaded model:** `{d['model_class']}`  · params: {_pc}  · new tokens: **{_nt}** (≤ {gen_params_used.max_new_tokens})
                - **raw model output (pre-trim):** <span style='color:#555'>{d['continuation_raw']}</span>
                - **final one-sentence continuation:** <span style='background:#e6f0ff'>{d['continuation']}</span>
                - **full text (prompt + continuation):** {d['full_text']}
                - **perplexity (full text):** {_ppl}
                """
            )
        )
    return mo.vstack(blocks)

_render()

## Cell 5 — STAGE 3 · GEC (per source)

The GEC model (`grammarly/coedit-large`, `method="dedicated"`, beam search) is loaded
**once** and reused. For each source we correct `full_text = prompt + " " + continuation`.
The corrected text is **what the pipeline treats as the error-free target** — the
difference between original and corrected is exactly where the errors are.

In [ ]:
# Load the corrector ONCE, reuse for every source.
gec_config = GECConfig()  # defaults: dedicated, grammarly/coedit-large, num_beams=4
_corrector = load_gec_corrector(gec_config, DEVICE)

gec = {}
for _name, _d in sources.items():
    _corrected = _corrector.correct([_d["full_text"]])[0]
    gec[_name] = {"full_text": _d["full_text"], "corrected": _corrected}

del _corrector
_msg = mo.md(f"GEC corrector `{gec_config.model_id}` loaded once; corrected {len(gec)} sources.")
_msg

In [ ]:
def _render():
    cfg = kv_table(
        "GECConfig (this stage)",
        {"method": gec_config.method, "model_id": gec_config.model_id, "num_beams": 4, "device": "auto"},
    )
    blocks = [cfg]
    for _name in ["authentic_learner", "matched_control", "artificial_learner"]:
        if _name not in gec:
            continue
        d = gec[_name]
        blocks.append(
            mo.vstack(
                [
                    mo.md(f"### {SOURCE_LABELS[_name]}"),
                    mo.md("**original → corrected (word-level diff):**"),
                    as_html(word_diff_html(d["full_text"], d["corrected"])),
                    mo.md(f"**corrected (error-free target):** {d['corrected']}"),
                ]
            )
        )
    return mo.vstack(blocks)

_render()

## Cell 6 — STAGE 4 · ERRANT (per source, region-classified)

ERRANT tags every correction edit. We annotate the **full text** then call
`classify_errors_by_region([ann], [len(prompt)])`: edits at char < boundary are
**prompt-region** (shared, *excluded*, greyed) and char ≥ boundary are
**generation-region** (the source's own, *counted*, solid). The per-source
`generation_error_type_counts` is the error-tag distribution every comparison uses.

In [ ]:
# Load the ERRANT annotator ONCE and reuse across sources.
errant_annotator = ERRANTAnnotator("en")

In [ ]:
annos = {}
gen_tag_counts = {}
for _name, _d in gec.items():
    _ann = errant_annotator.annotate_pair(_d["full_text"], _d["corrected"])
    classify_errors_by_region([_ann], [boundary])
    annos[_name] = _ann
    gen_tag_counts[_name] = dict(_ann.generation_error_type_counts)

_acfg = AnnotationConfig(lang="en")
ann_cfg_table = kv_table("AnnotationConfig (this stage)", {"lang": _acfg.lang, "prompt_char_boundary": boundary})

In [ ]:
def _edit_table(ann):
    rows = [
        "<tr><th>orig tokens</th><th>→ corr tokens</th><th>ERRANT tag</th>"
        "<th>char span</th><th>region</th></tr>"
    ]
    if not ann.errors:
        rows.append("<tr><td colspan=5><i>no edits (GEC left it unchanged)</i></td></tr>")
    for e in ann.errors:
        counted = e.region == "generation"
        bg = "#ffffff" if counted else "#efefef"
        col = "#111" if counted else "#999"
        tag_style = "font-weight:700" if counted else "font-weight:400"
        rows.append(
            f"<tr style='background:{bg};color:{col}'>"
            f"<td>{esc(e.original_tokens) or '∅'}</td>"
            f"<td>{esc(e.corrected_tokens) or '∅'}</td>"
            f"<td style='{tag_style}'>{esc(e.error_type)}</td>"
            f"<td>{e.char_start}–{e.char_end}</td>"
            f"<td>{esc(e.region)}{' ✓counted' if counted else ' (excluded)'}</td></tr>"
        )
    return (
        "<table style='border-collapse:collapse' border=1 cellpadding=4>"
        + "".join(rows)
        + "</table>"
    )

def _render():
    blocks = [
        ann_cfg_table,
        mo.md(
            f"*Grey rows = prompt-region (char < {boundary}, excluded). "
            f"Solid rows = generation-region (char ≥ {boundary}, counted).*"
        ),
    ]
    for _name in ["authentic_learner", "matched_control", "artificial_learner"]:
        if _name not in annos:
            continue
        ann = annos[_name]
        _dist = gen_tag_counts[_name]
        _dist_str = ", ".join(f"`{t}`×{n}" for t, n in sorted(_dist.items())) or "*(none)*"
        blocks.append(
            mo.vstack(
                [
                    mo.md(f"### {SOURCE_LABELS[_name]}"),
                    as_html(_edit_table(ann)),
                    mo.md(
                        f"**generation-region error-tag distribution** "
                        f"(`ann.generation_error_type_counts`): {_dist_str}"
                    ),
                ]
            )
        )
    return mo.vstack(blocks)

_render()

## Cell 7 — STAGE 5 · comparison against the authentic learner

The payoff. One column per source; rows for the corrected text, the generation-region
error-tag distribution, the generation-region error count, and the six **acquisition
categories**. Then a grouped bar chart, and a reading phrased as *distance / closeness to
the authentic learner* — the yardstick.

In [ ]:
def _present():
    return [n for n in ["authentic_learner", "matched_control", "artificial_learner"] if n in gen_tag_counts]

def _comparison_table():
    cols = _present()
    head = "<tr><th>row</th>" + "".join(f"<th>{esc(SOURCE_LABELS[c])}</th>" for c in cols) + "</tr>"
    rows = [head]

    # corrected full text
    rows.append(
        "<tr><td><b>corrected full text</b></td>"
        + "".join(f"<td style='max-width:260px'>{esc(gec[c]['corrected'])}</td>" for c in cols)
        + "</tr>"
    )
    # generation-region tag distribution
    def _dist_cell(c):
        d = gen_tag_counts[c]
        return ", ".join(f"{esc(t)}×{n}" for t, n in sorted(d.items())) or "—"

    rows.append(
        "<tr><td><b>generation-region error-tag counts</b></td>"
        + "".join(f"<td>{_dist_cell(c)}</td>" for c in cols)
        + "</tr>"
    )
    # generation-region error count
    rows.append(
        "<tr><td><b>errors in generation region (count)</b></td>"
        + "".join(f"<td>{sum(gen_tag_counts[c].values())}</td>" for c in cols)
        + "</tr>"
    )
    # acquisition categories
    for cat, name in ACQUISITION_CATS:
        cells = []
        for c in cols:
            cells.append(str(acquisition_view(gen_tag_counts[c]).get(cat, 0)))
        rows.append(
            f"<tr><td><code>{esc(cat)}</code> <span style='color:#666'>({esc(name)})</span></td>"
            + "".join(f"<td>{v}</td>" for v in cells)
            + "</tr>"
        )
    return (
        "<table style='border-collapse:collapse' border=1 cellpadding=5>"
        + "".join(rows)
        + "</table>"
    )

def _reading():
    cols = _present()
    learner = set(gen_tag_counts.get("authentic_learner", {}))
    control = set(gen_tag_counts.get("matched_control", {}))
    if "artificial_learner" not in cols:
        return mo.callout(
            mo.md(
                "**AL leg not available on this run** — so closeness *to the authentic learner* "
                "can't be measured here. Compare the **matched control** vs the **authentic "
                "learner** columns above: shared generation-region tags are where even the "
                "matched control already overlaps the yardstick. Enable the AL leg (Cell 3) to see "
                "whether fine-tuning moves the distribution *toward* the authentic learner beyond this."
            ),
            kind="info",
        )
    al = set(gen_tag_counts.get("artificial_learner", {}))
    toward = sorted((learner & al) - control)
    shared_all = sorted(learner & al & control)
    _t = ", ".join(f"`{t}`" for t in toward) or "*(none on this single sentence)*"
    _s = ", ".join(f"`{t}`" for t in shared_all) or "*(none)*"
    return mo.md(
        f"""
        **Reading — closeness to the authentic learner (the yardstick):**

        - Generation-region tags the **AL shares with the authentic learner that the control
          lacks** (the "moves *toward* the authentic learner" signal): {_t}
        - Tags shared by all three (overlap even native pretraining reaches): {_s}

        On a single sentence this is at most a handful of tags — *illustrative of the
        mechanism*, not a measurement. The claim `JSD(AL, LEARNER) < JSD(CONTROL, LEARNER)` is
        distributional over many sentences (see the caveat below).
        """
    )

mo.vstack([as_html(_comparison_table()), _reading()])

In [ ]:
def _chart():
    cols = [n for n in ["authentic_learner", "matched_control", "artificial_learner"] if n in gen_tag_counts]
    all_tags = sorted({t for c in cols for t in gen_tag_counts[c]})
    if not all_tags:
        return mo.md("*No generation-region error tags on any source for this sentence — nothing to plot.*")

    import numpy as np

    x = np.arange(len(all_tags))
    width = 0.8 / max(len(cols), 1)
    palette = {
        "authentic_learner": "#d1495b",  # yardstick — distinct warm
        "matched_control": "#8d99ae",  # matched control (native-pretrained), muted
        "artificial_learner": "#2a9d8f",  # fine-tuned — teal
    }
    fig, ax = plt.subplots(figsize=(max(6, 1.1 * len(all_tags)), 4))
    for i, c in enumerate(cols):
        counts = [gen_tag_counts[c].get(t, 0) for t in all_tags]
        ax.bar(x + i * width, counts, width, label=SOURCE_LABELS[c], color=palette.get(c, None))
    ax.set_xticks(x + width * (len(cols) - 1) / 2)
    ax.set_xticklabels(all_tags, rotation=40, ha="right", fontsize=8)
    ax.set_ylabel("generation-region count")
    ax.set_title("Generation-region error-tag counts — authentic learner vs control vs AL (n=1)")
    ax.legend(fontsize=8)
    ax.margins(y=0.15)
    fig.tight_layout()
    return fig

_chart()

## Cell 8 — Honest n=1 caveat (required)

In [ ]:
mo.callout(
    mo.md(
        """
        **This notebook is a single-example DEBUGGER for ownership — not evidence.**

        On `n=1`, a "distribution" is just a handful of tags: **illustrative only**. The
        central claim — `JSD(AL, LEARNER) < JSD(CONTROL, LEARNER)`, concentrated in
        acquisition categories — and its effect sizes (**JSD**, **RDR**) are **distributional
        over many sentences**. A single sentence can neither support nor refute it.

        For the real, pre-registered measurement run the **batch pipeline**:

        ```
        python -m gen_gec_errant.pipeline --config <config.yaml>
        ```

        and see `gen-gec-review-specs/` (claim, pre-registration, matrix, analyses) and
        `analysis-outputs/`. Do not read one sentence as if it settled anything.
        """
    ),
    kind="warn",
)